# ALL — Java Threading & Concurrency (Consolidated)

Master-level tour of parallel programming in Java: threads, the memory model, locks, atomics, the executor framework, `CompletableFuture`, fork/join, concurrent collections, synchronizers, and virtual threads (Loom) — each with runnable demos and real-world tasks.

> **Kernel note:** demos use `join()`/latches so output is deterministic. Concurrency bugs are timing-dependent — re-run cells a few times to observe interleavings.

> Prefer high-level constructs (executors, `CompletableFuture`, concurrent collections) over raw `Thread` + `synchronized` in real code.

# Overview — Concurrency vs Parallelism

## Foundations

| Term | Meaning |
|------|---------|
| **Process** | Isolated memory, own heap; heavy |
| **Thread** | Shares process heap; light; unit of scheduling |
| **Concurrency** | *Dealing with* many things (interleaving) |
| **Parallelism** | *Doing* many things at once (multi-core) |
| **Race condition** | Result depends on timing of unsynchronized access |
| **Critical section** | Code that must run mutually exclusive |

```mermaid
flowchart TB
    P[JVM Process] --> H[Shared Heap objects]
    P --> T1[Thread 1 stack]
    P --> T2[Thread 2 stack]
    P --> T3[Thread 3 stack]
    H -.shared mutable state.-> T1
    H -.needs synchronization.-> T2
```

### The core problems
1. **Visibility** — one thread's write may not be seen by another (caches, reordering) → `volatile`, locks.
2. **Atomicity** — compound ops (`i++`) aren't atomic → atomics/locks.
3. **Ordering** — compiler/CPU reorder instructions → happens-before via JMM.

### Golden rules
- Share **immutable** data, or confine mutable data to one thread, or synchronize all access.
- Prefer `java.util.concurrent` over hand-rolled locking.

##### Tasks — Concurrency Fundamentals

> Mix of core mechanics · design · real-world backend scenarios.

1. Write a program with two threads incrementing a shared `int` 100k times each without synchronization; show the final value is `< 200000` (lost updates).
2. Fix the above with `synchronized`, then with `AtomicInteger`; compare.
3. Demonstrate a **visibility** bug: a `boolean running` flag toggled by one thread that another never sees; fix with `volatile`.
4. **Real-world** — Explain (in comments) whether a stateless request handler needs synchronization and why immutability/thread-confinement removes the need.


# Creating & Running Threads

## Thread vs Runnable vs Callable

| Approach | Notes |
|----------|-------|
| `extends Thread`, override `run()` | Couples task to thread; rarely preferred |
| `implements Runnable` → `new Thread(r)` | Preferred; task/execution separated |
| lambda `Runnable` | Concise |
| `Callable<V>` + `ExecutorService.submit` | Returns a value / throws checked exceptions |

- `start()` spawns a new thread and calls `run()`; **calling `run()` directly runs on the current thread** (common bug).
- Always name threads for debugging; set uncaught exception handlers.

In [ ]:
// Demo: Runnable + join
Runnable task = () -> {
    for (int i = 0; i < 3; i++) System.out.println(Thread.currentThread().getName() + " tick " + i);
};
Thread t = new Thread(task, "worker-1");
t.start();
t.join();               // wait for completion (deterministic output)
System.out.println("done on " + Thread.currentThread().getName());


##### Tasks — Creating Threads

> Mix of core mechanics · design · real-world backend scenarios.

1. Launch 5 threads that each print their name; `join` them all; confirm main waits.
2. Show the difference between `t.start()` and `t.run()` (thread name printed).
3. Set a custom `Thread.UncaughtExceptionHandler` and throw inside a thread to see it fire.
4. Create a daemon thread and show the JVM can exit without waiting for it.
5. **Real-world** — Implement a `NamedThreadFactory` (prefix + counter + daemon flag) you'd hand to an executor.


# Thread Lifecycle & Control Methods

## States and transitions

```mermaid
stateDiagram-v2
    [*] --> NEW
    NEW --> RUNNABLE: start()
    RUNNABLE --> BLOCKED: wait for monitor lock
    RUNNABLE --> WAITING: wait() / join() / park()
    RUNNABLE --> TIMED_WAITING: sleep(t) / wait(t)
    BLOCKED --> RUNNABLE: lock acquired
    WAITING --> RUNNABLE: notify() / target done
    TIMED_WAITING --> RUNNABLE: timeout
    RUNNABLE --> TERMINATED: run() returns
    TERMINATED --> [*]
```

| Method | Effect |
|--------|--------|
| `start()` | NEW → RUNNABLE |
| `join()` | Caller waits for the thread to finish |
| `sleep(ms)` | TIMED_WAITING; keeps locks! |
| `interrupt()` | Sets interrupt flag; wakes blocking calls with `InterruptedException` |
| `isInterrupted()` / `Thread.interrupted()` | Check (latter clears flag) |
| `yield()` | Hint to scheduler (rarely useful) |
| `setDaemon(true)` | JVM won't wait for it on exit |
| `setPriority(n)` | Hint only (1–10) |

**Interruption is cooperative** — code must check the flag / handle `InterruptedException` and stop; never use the removed `stop()`.

In [ ]:
// Demo: cooperative cancellation via interrupt
Thread worker = new Thread(() -> {
    long n = 0;
    while (!Thread.currentThread().isInterrupted()) {
        n++;
        if (n % 20_000_000 == 0) System.out.println("working... " + n);
        if (n > 60_000_000) break;
    }
    System.out.println("worker exiting, interrupted=" + Thread.currentThread().isInterrupted());
});
worker.start();
Thread.sleep(50);
worker.interrupt();      // ask it to stop
worker.join();
System.out.println("main done");


##### Tasks — Lifecycle & Control

> Mix of core mechanics · design · real-world backend scenarios.

1. Print `Thread.getState()` transitions by observing a thread through NEW→RUNNABLE→TIMED_WAITING→TERMINATED.
2. Implement a worker that stops cleanly on `interrupt()` while sleeping (catch `InterruptedException`, restore flag, exit).
3. Show that `sleep` does NOT release a held lock (two threads, one sleeps inside `synchronized`).
4. Build a `Stopwatch` thread that can be paused/resumed via a flag + `wait`/`notify`.
5. **Real-world** — Implement a graceful-shutdown pattern: a long-running worker loop that checks interruption between units of work and flushes state before exiting.


# Runnable vs Callable & Future

## Returning results

| | Runnable | Callable&lt;V&gt; |
|---|----------|-------------------|
| Returns value | No (`void`) | Yes (`V`) |
| Throws checked | No | Yes |
| Submit to executor | `execute`/`submit` | `submit` → `Future<V>` |

`Future<V>`: `get()` (blocks), `get(timeout)`, `isDone()`, `cancel(mayInterrupt)`, `isCancelled()`.

In [ ]:
// Demo: Callable + Future
import java.util.concurrent.*;
ExecutorService ex = Executors.newFixedThreadPool(2);
Future<Integer> f = ex.submit(() -> { Thread.sleep(30); return 6 * 7; });
System.out.println("doing other work...");
System.out.println("result = " + f.get());   // blocks until ready
ex.shutdown();


##### Tasks — Callable & Future

> Mix of core mechanics · design · real-world backend scenarios.

1. Submit 3 `Callable`s that compute squares; collect results via `Future.get()` in order.
2. Use `invokeAll` to run a batch and gather all results; then `invokeAny` to get the first.
3. Demonstrate `Future.cancel(true)` interrupting a running task.
4. Add a timeout with `future.get(timeout)` and handle `TimeoutException`.
5. **Real-world** — Fan out N independent remote calls (simulate with sleep) and aggregate results, failing fast if any errors.


# Synchronization — synchronized & Monitors

## Intrinsic locks (monitors)

- Every object has an intrinsic lock (monitor). `synchronized` method/block acquires it.
- `synchronized` instance method → locks `this`; `static synchronized` → locks the `Class` object.
- Only **one** thread holds a given monitor at a time → mutual exclusion + establishes happens-before (visibility).
- Keep critical sections small; never call foreign/blocking code holding a lock.

```mermaid
sequenceDiagram
    participant T1
    participant T2
    participant Lock as Monitor
    T1->>Lock: acquire
    Note over T1: critical section
    T2->>Lock: acquire (BLOCKED)
    T1->>Lock: release
    Lock-->>T2: granted
```

In [ ]:
// Demo: synchronized fixes lost updates
class Counter {
    private int value;
    synchronized void inc() { value++; }
    synchronized int get() { return value; }
}
Counter counter = new Counter();
Runnable r = () -> { for (int i = 0; i < 100_000; i++) counter.inc(); };
Thread a = new Thread(r), b = new Thread(r);
a.start(); b.start(); a.join(); b.join();
System.out.println("value = " + counter.get() + " (expect 200000)");


##### Tasks — Synchronization

> Mix of core mechanics · design · real-world backend scenarios.

1. Reproduce lost updates without sync, then fix with a `synchronized` method and with a `synchronized(lockObj)` block.
2. Show the difference between locking on `this` vs a private final lock object (avoid locking on public objects/strings).
3. Demonstrate `static synchronized` locks the class, not the instance (two instances still serialize).
4. Build a thread-safe bounded counter with `synchronized` + `wait`/`notify` blocking when at capacity.
5. **Real-world** — Implement a thread-safe in-memory sequence generator and a thread-safe simple cache using `synchronized`; note the contention hotspot.


# volatile, Visibility & the Java Memory Model

## Happens-before

- **`volatile`** guarantees visibility + prevents reordering around the access, but is **not atomic** for compound ops (`v++`).
- **Happens-before** edges: program order within a thread; `unlock→lock` on same monitor; `volatile write→read`; `Thread.start()→run`; `run end→join`; `Executor.submit→task`.
- Use `volatile` for **flags** and single-writer publish; use atomics/locks for read-modify-write.

| Need | Tool |
|------|------|
| Visible flag | `volatile boolean` |
| Counter / CAS | `AtomicInteger`/`AtomicLong` |
| Compound invariant across fields | lock |
| Safe publication of immutable object | `final` fields / `volatile` reference |

In [ ]:
// Demo: volatile flag for safe stop
class Task implements Runnable {
    volatile boolean running = true;      // without volatile, may loop forever
    public void run() {
        long n = 0;
        while (running) n++;
        System.out.println("stopped after " + n + " iters");
    }
}
Task task = new Task();
Thread t = new Thread(task);
t.start();
Thread.sleep(20);
task.running = false;    // visible thanks to volatile
t.join();


##### Tasks — volatile & JMM

> Mix of core mechanics · design · real-world backend scenarios.

1. Show a stop-flag loop that (may) hang without `volatile` and stops reliably with it.
2. Prove `volatile v++` still loses updates under contention; switch to `AtomicInteger`.
3. Implement the double-checked-locking singleton correctly with a `volatile` instance field; explain why `volatile` is required.
4. Demonstrate safe publication of an immutable config object via a `volatile` reference (swap whole object atomically).
5. **Real-world** — Build a hot-reloadable configuration holder: readers see either the old or new immutable snapshot, never a torn state.


# Atomic Classes & CAS

## Lock-free primitives

- `AtomicInteger/Long/Boolean/Reference` use **CAS** (compare-and-set) — lock-free, no blocking.
- Key ops: `get`, `set`, `incrementAndGet`, `getAndAdd`, `compareAndSet(expect,update)`, `updateAndGet(fn)`, `accumulateAndGet`.
- `LongAdder`/`DoubleAdder`: better than `AtomicLong` under **high contention** (striped counters).
- `AtomicReference` for lock-free stacks/state machines; beware the **ABA** problem (`AtomicStampedReference`).

In [ ]:
// Demo: AtomicInteger + CAS loop
import java.util.concurrent.atomic.*;
AtomicInteger counter = new AtomicInteger();
Runnable r = () -> { for (int i = 0; i < 100_000; i++) counter.incrementAndGet(); };
Thread a = new Thread(r), b = new Thread(r);
a.start(); b.start(); a.join(); b.join();
System.out.println("atomic = " + counter.get());

// CAS-based max update
AtomicInteger max = new AtomicInteger(0);
int candidate = 42;
int cur;
while ((cur = max.get()) < candidate && !max.compareAndSet(cur, candidate)) { }
System.out.println("max = " + max.get());


##### Tasks — Atomics & CAS

> Mix of core mechanics · design · real-world backend scenarios.

1. Replace a `synchronized` counter with `AtomicInteger`; benchmark conceptually.
2. Implement an atomic 'update max' and an atomic 'multiply' using `updateAndGet`/`accumulateAndGet`.
3. Build a **lock-free Treiber stack** with `AtomicReference` (push/pop via CAS).
4. Demonstrate the ABA problem and fix it with `AtomicStampedReference`.
5. Compare `AtomicLong` vs `LongAdder` under many threads incrementing a shared counter.
6. **Real-world** — Implement a lock-free request-id generator and a metrics counter (hits) using `LongAdder`.


# Explicit Locks — ReentrantLock, ReadWriteLock, StampedLock

## When intrinsic locks aren't enough

| Lock | Superpower |
|------|-----------|
| `ReentrantLock` | `tryLock`, timed lock, interruptible, fairness, multiple `Condition`s |
| `ReentrantReadWriteLock` | Many readers **or** one writer |
| `StampedLock` | Optimistic reads (fastest for read-heavy), non-reentrant |

- Always unlock in `finally`.
- `Condition` replaces `wait/notify` with multiple wait-sets (`await`/`signal`/`signalAll`).
- Fairness reduces starvation but lowers throughput.

In [ ]:
// Demo: ReentrantLock with tryLock + Condition
import java.util.concurrent.locks.*;
ReentrantLock lock = new ReentrantLock();
int[] shared = {0};
Runnable r = () -> {
    if (lock.tryLock()) {
        try { shared[0]++; }
        finally { lock.unlock(); }
    } else {
        System.out.println(Thread.currentThread().getName() + " skipped (lock busy)");
    }
};
Thread a = new Thread(r, "A"), b = new Thread(r, "B");
a.start(); b.start(); a.join(); b.join();
System.out.println("shared = " + shared[0]);


##### Tasks — Explicit Locks

> Mix of core mechanics · design · real-world backend scenarios.

1. Rewrite a `synchronized` counter using `ReentrantLock` with unlock in `finally`.
2. Use `tryLock(timeout)` to avoid indefinite blocking; log when acquisition times out.
3. Build a thread-safe cache with `ReentrantReadWriteLock` (concurrent reads, exclusive writes).
4. Implement a bounded blocking buffer using `ReentrantLock` + two `Condition`s (`notFull`, `notEmpty`).
5. Use `StampedLock` optimistic read for a rarely-changing coordinate pair; validate and fall back to a read lock.
6. **Real-world / Big task** — Build a read-heavy in-memory routing table (prefix → handler) with RW-lock; measure that readers don't block each other.


# wait / notify — Producer / Consumer

## The classic coordination primitive

- `wait()`, `notify()`, `notifyAll()` must be called **while holding the object's monitor** (`synchronized`).
- Always `wait` in a **while loop** re-checking the condition (guard against spurious wakeups & missed signals).
- Prefer `notifyAll` unless you can prove `notify` is safe.

**Modern replacement:** `BlockingQueue` — don't hand-roll this in production.

In [ ]:
// Demo: bounded buffer with wait/notify
import java.util.*;
class Buffer {
    private final Queue<Integer> q = new LinkedList<>();
    private final int cap;
    Buffer(int cap){ this.cap = cap; }
    synchronized void put(int x) throws InterruptedException {
        while (q.size() == cap) wait();
        q.add(x); notifyAll();
    }
    synchronized int take() throws InterruptedException {
        while (q.isEmpty()) wait();
        int x = q.poll(); notifyAll(); return x;
    }
}
Buffer buf = new Buffer(2);
Thread producer = new Thread(() -> { try { for (int i=1;i<=5;i++){ buf.put(i); System.out.println("put "+i);} } catch(Exception e){} });
Thread consumer = new Thread(() -> { try { for (int i=1;i<=5;i++){ System.out.println("take "+buf.take()); Thread.sleep(10);} } catch(Exception e){} });
producer.start(); consumer.start(); producer.join(); consumer.join();


##### Tasks — wait / notify

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement a one-slot handoff (`SynchronousQueue`-like) using `wait`/`notify`.
2. Build a bounded blocking queue with `wait`/`notify` supporting multiple producers/consumers (use `notifyAll`).
3. Show a missed-signal bug using `if` instead of `while`, then fix it.
4. Implement a simple `CountDownLatch` from scratch using `wait`/`notify`.
5. **Real-world / Big task** — Build a thread pool from scratch: a fixed set of worker threads pulling `Runnable`s from a hand-rolled blocking queue; support graceful `shutdown()`.


# Deadlock, Livelock & Starvation

## Liveness hazards

```mermaid
flowchart LR
    T1 -->|holds A, wants B| B
    T2 -->|holds B, wants A| A
    A -->|held by T1| T1
    B -->|held by T2| T2
```

| Hazard | Cause | Fix |
|--------|-------|-----|
| **Deadlock** | Circular lock wait | Global lock ordering; `tryLock` with backoff; single lock |
| **Livelock** | Threads react to each other, no progress | Randomized backoff |
| **Starvation** | A thread never gets CPU/lock | Fair locks; avoid priority abuse |

**Prevent deadlock:** acquire locks in a consistent global order; hold as few locks as possible; use timeouts.

In [ ]:
// Demo: deadlock avoidance via lock ordering
import java.util.concurrent.locks.*;
ReentrantLock l1 = new ReentrantLock(), l2 = new ReentrantLock();
// Always acquire l1 before l2 in BOTH threads -> no cycle
Runnable safe = () -> {
    l1.lock();
    try { l2.lock();
        try { System.out.println(Thread.currentThread().getName() + " did work"); }
        finally { l2.unlock(); }
    } finally { l1.unlock(); }
};
Thread a = new Thread(safe, "A"), b = new Thread(safe, "B");
a.start(); b.start(); a.join(); b.join();
System.out.println("no deadlock");


##### Tasks — Liveness Hazards

> Mix of core mechanics · design · real-world backend scenarios.

1. Deliberately create a two-lock deadlock (opposite ordering); detect it via a thread dump (`jstack`) — describe what you'd see.
2. Fix the deadlock by imposing a global lock ordering.
3. Fix the same scenario instead using `tryLock` with timeout + backoff (avoid holding one lock while waiting).
4. Reproduce a livelock (two 'polite' threads repeatedly yielding) and fix with randomized backoff.
5. **Real-world** — Two accounts, concurrent transfers in both directions; make `transfer(a,b,amt)` deadlock-free by locking accounts in id order.


# Executor Framework & Thread Pools

## Don't manage threads by hand

```mermaid
flowchart LR
    Q[Task Queue] --> P[Thread Pool workers]
    P --> C[reused threads]
    subgraph ExecutorService
      Q
      P
    end
```

| Factory | Pool |
|---------|------|
| `newFixedThreadPool(n)` | Fixed size, unbounded queue |
| `newCachedThreadPool()` | Grows/shrinks, 60s idle TTL |
| `newSingleThreadExecutor()` | Serial execution |
| `newScheduledThreadPool(n)` | Delayed/periodic tasks |
| `newVirtualThreadPerTaskExecutor()` | Java 21 virtual threads |

**Prefer configuring `ThreadPoolExecutor` directly** (core/max size, bounded queue, `ThreadFactory`, `RejectedExecutionHandler`) to control backpressure.

**Lifecycle:** `submit`/`execute` → `shutdown()` (graceful) or `shutdownNow()` → `awaitTermination(timeout)`.

In [ ]:
// Demo: ThreadPoolExecutor with bounded queue + naming
import java.util.concurrent.*;
import java.util.concurrent.atomic.AtomicInteger;
AtomicInteger seq = new AtomicInteger();
ThreadPoolExecutor pool = new ThreadPoolExecutor(
    2, 4, 30, TimeUnit.SECONDS,
    new ArrayBlockingQueue<>(10),
    r -> { Thread t = new Thread(r, "pool-" + seq.incrementAndGet()); return t; },
    new ThreadPoolExecutor.CallerRunsPolicy());
for (int i = 0; i < 6; i++) {
    int id = i;
    pool.submit(() -> System.out.println("task " + id + " on " + Thread.currentThread().getName()));
}
pool.shutdown();
pool.awaitTermination(2, TimeUnit.SECONDS);
System.out.println("all tasks complete");


##### Tasks — Executors & Pools

> Mix of core mechanics · design · real-world backend scenarios.

1. Run 100 tasks on a fixed pool of 4; confirm only 4 threads are reused (log thread names).
2. Configure a `ThreadPoolExecutor` with a bounded queue and a `RejectedExecutionHandler`; overflow it and observe rejection/caller-runs.
3. Schedule a periodic heartbeat with `newScheduledThreadPool` (`scheduleAtFixedRate`); cancel after N ticks.
4. Implement graceful shutdown: `shutdown()` + `awaitTermination`, falling back to `shutdownNow()`.
5. **Real-world** — Build a bounded worker pool that processes an incoming queue of 'orders' with backpressure and a metrics summary (processed, rejected).
6. **Big task** — Implement a tiny `ThreadPoolExecutor` clone: N workers, bounded task queue, submit/shutdown/awaitTermination, and a rejection policy.


# CompletableFuture — Async Composition

## Non-blocking pipelines

| Method | Use |
|--------|-----|
| `supplyAsync(fn)` / `runAsync` | Start async work |
| `thenApply` / `thenApplyAsync` | Transform result |
| `thenCompose` | Chain another future (flatMap) |
| `thenCombine` | Combine two futures |
| `allOf` / `anyOf` | Wait for all / first |
| `exceptionally` / `handle` / `whenComplete` | Error handling |
| `orTimeout` (Java 9+) | Fail on timeout |

**Tip:** pass your own `Executor` to `*Async` variants; the default is the common `ForkJoinPool`.
**Don't** block inside pipeline stages (no `get()` mid-chain).

In [ ]:
// Demo: CompletableFuture pipeline + combine + error handling
import java.util.concurrent.*;
CompletableFuture<Integer> price = CompletableFuture.supplyAsync(() -> { sleep(20); return 100; });
CompletableFuture<Integer> tax   = CompletableFuture.supplyAsync(() -> { sleep(10); return 18; });
CompletableFuture<String> total = price.thenCombine(tax, (p, t) -> p + t)
        .thenApply(sum -> "total = " + sum)
        .exceptionally(ex -> "failed: " + ex.getMessage());
System.out.println(total.get());

// helper
static void sleep(long ms){ try { Thread.sleep(ms);} catch(InterruptedException e){ Thread.currentThread().interrupt(); } }


##### Tasks — CompletableFuture

> Mix of core mechanics · design · real-world backend scenarios.

1. Chain `supplyAsync → thenApply → thenAccept`; run on a custom executor.
2. Use `thenCompose` to chain two dependent async calls (fetch user → fetch orders).
3. Use `thenCombine` to merge two independent calls; then `allOf` to await a list of futures and collect results.
4. Add robust error handling with `handle`/`exceptionally` and a timeout via `orTimeout`.
5. **Real-world** — Scatter-gather: call 5 pricing services in parallel, take the min, fall back to a default on failure/timeout.
6. **Big task** — Build an async pipeline with retry+backoff (compose `CompletableFuture`s) and a circuit-breaker-lite that opens after N failures.


# Fork/Join & Parallel Streams

## Divide and conquer

- `ForkJoinPool` with **work-stealing**: idle workers steal tasks from busy queues' tails.
- `RecursiveTask<V>` (returns) / `RecursiveAction` (void): `fork()` + `join()`; split until a threshold, then compute directly.
- **Parallel streams** (`.parallel()`) run on the common `ForkJoinPool` — great for CPU-bound, side-effect-free, splittable work; bad for I/O or shared mutable state.

In [ ]:
// Demo: parallel sum via RecursiveTask
import java.util.concurrent.*;
class SumTask extends RecursiveTask<Long> {
    final long[] a; final int lo, hi;
    SumTask(long[] a, int lo, int hi){ this.a=a; this.lo=lo; this.hi=hi; }
    protected Long compute() {
        if (hi - lo <= 1000) { long s=0; for (int i=lo;i<hi;i++) s+=a[i]; return s; }
        int mid = (lo+hi)>>>1;
        SumTask left = new SumTask(a, lo, mid); left.fork();
        SumTask right = new SumTask(a, mid, hi);
        long r = right.compute();
        return left.join() + r;
    }
}
long[] data = new long[100_000];
for (int i=0;i<data.length;i++) data[i]=i;
System.out.println("sum = " + new ForkJoinPool().invoke(new SumTask(data,0,data.length)));


##### Tasks — Fork/Join & Parallel Streams

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement parallel array sum and parallel max with `RecursiveTask` and a sensible split threshold.
2. Implement parallel merge sort with `RecursiveAction`.
3. Compare a `for` loop vs `IntStream.range().parallel()` for a CPU-bound computation (correctness + rough timing).
4. Show a parallel-stream bug from shared mutable state; fix with a reduction/collector.
5. **Real-world / Big task** — Parallelize processing of a large list of records (map→filter→reduce) with a parallel stream; ensure the accumulator is associative & stateless.


# Concurrent Collections

## Thread-safe data structures

| Collection | Use |
|------------|-----|
| `ConcurrentHashMap` | High-concurrency map; atomic `compute`/`merge`/`putIfAbsent` |
| `CopyOnWriteArrayList` | Read-heavy, rare writes (listeners) |
| `ConcurrentLinkedQueue` | Lock-free unbounded queue |
| `BlockingQueue` (`ArrayBlockingQueue`, `LinkedBlockingQueue`, `SynchronousQueue`, `PriorityBlockingQueue`, `DelayQueue`) | Producer/consumer with blocking put/take |
| `ConcurrentSkipListMap/Set` | Sorted + concurrent |

- Prefer these over `Collections.synchronizedX` (which locks the whole structure and needs manual sync for iteration).
- `ConcurrentHashMap` iteration is weakly consistent (no `ConcurrentModificationException`).

In [ ]:
// Demo: ConcurrentHashMap atomic merge + BlockingQueue
import java.util.concurrent.*;
ConcurrentHashMap<String,Integer> counts = new ConcurrentHashMap<>();
Runnable r = () -> { for (int i=0;i<1000;i++) counts.merge("hits", 1, Integer::sum); };
Thread a = new Thread(r), b = new Thread(r);
a.start(); b.start(); a.join(); b.join();
System.out.println("hits = " + counts.get("hits"));

BlockingQueue<Integer> q = new ArrayBlockingQueue<>(3);
q.put(1); q.put(2);
System.out.println("take = " + q.take() + ", size=" + q.size());


##### Tasks — Concurrent Collections

> Mix of core mechanics · design · real-world backend scenarios.

1. Build a thread-safe word-frequency counter with `ConcurrentHashMap.merge` across multiple threads.
2. Use `computeIfAbsent` to build a concurrent multimap (`Map<K, List<V>>`) safely.
3. Implement producer/consumer with `ArrayBlockingQueue` (no manual wait/notify).
4. Use `SynchronousQueue` for a direct handoff; contrast with `LinkedBlockingQueue`.
5. Use `PriorityBlockingQueue` for a priority task feed and `DelayQueue` for scheduled expiry.
6. **Real-world / Big task** — Build a bounded in-memory job pipeline: N producers → `BlockingQueue` → M consumers, with graceful drain on shutdown and metrics.


# Synchronizers — Latches, Barriers, Semaphores, Phaser, Exchanger

## Coordinating groups of threads

| Tool | Use |
|------|-----|
| `CountDownLatch` | Wait for N events once (start gate / done gate) |
| `CyclicBarrier` | N threads rendezvous repeatedly |
| `Semaphore` | Limit concurrent access to a resource (permits) |
| `Phaser` | Flexible, reusable multi-phase barrier |
| `Exchanger` | Two threads swap objects |

In [ ]:
// Demo: CountDownLatch (done gate) + Semaphore (limit)
import java.util.concurrent.*;
int n = 3;
CountDownLatch done = new CountDownLatch(n);
Semaphore permits = new Semaphore(2);  // at most 2 concurrent
for (int i = 0; i < n; i++) {
    int id = i;
    new Thread(() -> {
        try { permits.acquire(); System.out.println("worker "+id+" running"); Thread.sleep(20); }
        catch (InterruptedException e){ Thread.currentThread().interrupt(); }
        finally { permits.release(); done.countDown(); }
    }).start();
}
done.await();
System.out.println("all workers finished");


##### Tasks — Synchronizers

> Mix of core mechanics · design · real-world backend scenarios.

1. Use a `CountDownLatch` as a **start gate** so all threads begin simultaneously, and a second latch as a **done gate**.
2. Use a `CyclicBarrier` to run a multi-phase simulation where all threads must finish phase k before phase k+1.
3. Implement a connection-limiter with a `Semaphore(maxPermits)` (acquire/release around resource use).
4. Use an `Exchanger` for a producer/consumer buffer swap.
5. **Real-world / Big task** — Build a parallel map-reduce: split work with a start-gate latch, barrier between map and reduce phases, and a semaphore to bound resource usage.


# ThreadLocal & Thread Confinement

## Per-thread state

- `ThreadLocal<T>` gives each thread its own value — no sharing, no sync.
- Common uses: per-thread `SimpleDateFormat` (not thread-safe), request/trace context, DB connection per thread.
- **Danger:** leaks in thread pools — threads are reused, so **always `remove()`** in a `finally` after the request.
- `InheritableThreadLocal` passes values to child threads.

In [ ]:
// Demo: ThreadLocal per-thread context
ThreadLocal<String> ctx = ThreadLocal.withInitial(() -> "none");
Runnable r = () -> {
    ctx.set("req-" + Thread.currentThread().getName());
    try { System.out.println(Thread.currentThread().getName() + " sees " + ctx.get()); }
    finally { ctx.remove(); }   // avoid leak in pooled threads
};
Thread a = new Thread(r, "A"), b = new Thread(r, "B");
a.start(); b.start(); a.join(); b.join();


##### Tasks — ThreadLocal

> Mix of core mechanics · design · real-world backend scenarios.

1. Store a per-thread `SimpleDateFormat` in a `ThreadLocal` and format dates from multiple threads without corruption.
2. Build a lightweight per-request trace/correlation-id context using `ThreadLocal`; set at entry, `remove()` at exit.
3. Demonstrate a `ThreadLocal` leak in a reused pool thread (value persists to the next task) and fix with `remove()`.
4. Use `InheritableThreadLocal` and observe a child thread inheriting the parent's value.
5. **Real-world** — Implement an MDC-style logging context that tags every log line with the current request id.


# Virtual Threads & Structured Concurrency (Java 21)

## Project Loom

- **Virtual threads** are lightweight threads scheduled by the JVM onto a small pool of OS carrier threads. Millions are cheap.
- Blocking a virtual thread (I/O, `sleep`) **parks** it and frees the carrier — write simple blocking code at massive scale.
- Create via `Thread.ofVirtual().start(r)` or `Executors.newVirtualThreadPerTaskExecutor()`.
- **Structured concurrency** (`StructuredTaskScope`) ties subtask lifetimes to a scope — clean cancellation & error propagation.

| Platform thread | Virtual thread |
|-----------------|----------------|
| ~1MB stack, OS-scheduled | Tiny, JVM-scheduled |
| Thousands max | Millions |
| Pool to reuse | Don't pool — create per task |

> Avoid pinning: don't hold `synchronized` across blocking calls on virtual threads (use `ReentrantLock`).

In [ ]:
// Demo: virtual threads (Java 21). If your kernel is <21, read the code instead.
import java.util.concurrent.*;
try (ExecutorService ex = Executors.newVirtualThreadPerTaskExecutor()) {
    for (int i = 0; i < 5; i++) {
        int id = i;
        ex.submit(() -> {
            try { Thread.sleep(10); } catch (InterruptedException e) {}
            System.out.println("task " + id + " on " + Thread.currentThread());
        });
    }
} // close() waits for all tasks
System.out.println("done");


##### Tasks — Virtual Threads & Structured Concurrency

> Mix of core mechanics · design · real-world backend scenarios.

1. Launch 10,000 virtual threads that each sleep briefly and increment an `AtomicInteger`; confirm the count and note how cheap it is.
2. Compare a fixed platform-thread pool vs `newVirtualThreadPerTaskExecutor` for many blocking (sleep) tasks.
3. Demonstrate **pinning**: hold a `synchronized` block across a sleep on a virtual thread; then refactor to `ReentrantLock`.
4. Use `StructuredTaskScope.ShutdownOnFailure` to run two subtasks and fail fast if either errors (Java 21 preview).
5. **Real-world / Big task** — Build a concurrent 'fetch and aggregate' that issues 1,000 simulated I/O calls on virtual threads and aggregates results with structured concurrency + a timeout.


# Concurrency Patterns & Best Practices

## Reusable designs

```mermaid
flowchart LR
    subgraph Producer-Consumer
      P1[Producers] --> BQ[BlockingQueue] --> C1[Consumers]
    end
```

| Pattern | Essence |
|---------|---------|
| Producer/Consumer | Decouple via a blocking queue |
| Thread pool | Reuse workers, bound concurrency |
| Future/Promise | Represent async result |
| Fan-out / Fan-in | Split work, aggregate results |
| Pipeline | Stages connected by queues |
| Scatter-Gather | Parallel calls, combine |
| Read-Copy-Update | Swap immutable snapshots for readers |

### Best practices
1. Prefer immutability & thread confinement; minimize shared mutable state.
2. Use high-level `java.util.concurrent` over raw locks.
3. Keep critical sections tiny; never block/callback while holding a lock.
4. Always handle `InterruptedException` (restore the flag).
5. Bound your queues and pools → backpressure, no OOM.
6. Name threads; add uncaught-exception handlers; log context.
7. Test with stress/interleaving; reason with happens-before.

##### Tasks — Patterns & Best Practices

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement producer/consumer, then a multi-stage pipeline (stage1→queue→stage2→queue→stage3).
2. Implement fan-out/fan-in: split a task into N subtasks, run in parallel, aggregate.
3. Implement scatter-gather across services with per-call timeouts and partial-result tolerance.
4. Implement Read-Copy-Update: readers use a `volatile` immutable snapshot; writers replace it atomically.
5. **Real-world / Big task** — Build a mini job-processing system: bounded queue, worker pool, retries with backoff, dead-letter queue, metrics, and graceful shutdown — combining executors, concurrent collections, and atomics.


# Quick Reference Card

| Need | Reach for |
|------|-----------|
| Visible flag | `volatile boolean` |
| Counter under contention | `AtomicInteger` / `LongAdder` |
| Mutual exclusion | `synchronized` / `ReentrantLock` |
| Read-heavy sharing | `ReentrantReadWriteLock` / `StampedLock` |
| Producer/consumer | `BlockingQueue` |
| Run many tasks | `ExecutorService` (fixed / virtual) |
| Async composition | `CompletableFuture` |
| CPU divide-and-conquer | Fork/Join, parallel streams |
| Wait for N events | `CountDownLatch` |
| Repeated rendezvous | `CyclicBarrier` |
| Limit concurrency | `Semaphore` |
| Per-thread state | `ThreadLocal` (remember `remove()`) |
| Massive blocking I/O | Virtual threads |

---

✅ Threads, JMM, locks, atomics, executors, `CompletableFuture`, fork/join, concurrent collections, synchronizers, `ThreadLocal`, virtual threads, and patterns — all covered. Grind the tasks; re-run to observe interleavings.